<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/14%20-%20Menor%20Caminho%20Dijkstra%20e%20Roteamento%20Dinamico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 14 - Notebook: Algoritmo de Dijkstra e Roteamento Dinâmico em Tempo Real
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos o **Algoritmo de Dijkstra** utilizando fila de prioridade binária (*Min-Heap* via `heapq`) para o cálculo determinístico de rotas de custo métrico mínimo e reencaminhamento dinâmico em caso de bloqueios operacionais ou congestionamento na malha de esteiras da Célula de Manufatura Flexível.

In [1]:
import heapq
from typing import List, Dict, Set, Tuple, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[Any]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura alinhada."""
    larguras = [max(len(str(r)), 7) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else f"{v:.1f}"
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoManufatura:
    """
    Representa a malha física de transporte e triagem da FMS como Dígrafo Ponderado G = (V, E, W).
    """
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)

        # Matriz binária de adjacência (1 se há segmento direto u -> v, 0 caso contrário)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]

        # Matriz de adjacência ponderada por distâncias físicas (metros)
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0

        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_rota(self, origem: str, destino: str, comprimento_m: float,
                        tag_atuador: str, mecanismo: str = "Esteira"):
        """Adiciona um segmento de esteira ou rampa com sentido único e peso associado."""
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador ISA": tag_atuador,
            "Mecanismo": mecanismo
        })

    def obter_atuador_aresta(self, u: str, v: str) -> str:
        """Retorna a tag ISA do atuador responsável pela transição entre os postos u e v."""
        for a in self.arestas_detalhes:
            if a["Origem"] == u and a["Destino"] == v:
                return a["Atuador ISA"]
        return "N/A"

    def obter_peso_aresta(self, u: str, v: str) -> float:
        """Retorna o comprimento físico do segmento em metros."""
        u_idx = self.v_to_idx[u]
        v_idx = self.v_to_idx[v]
        return self.adj_pesos[u_idx][v_idx]

def criar_planta_fms_nominal() -> GrafoManufatura:
    """Instancia a topologia nominal da Célula FMS (linha linear-ramificada padrão)."""
    postos = [
        "SILO-100", "INSP-200", "DIV-201", "DIV-202", "DIV-203",
        "CX-301", "CX-302", "CX-303", "DESC-304"
    ]
    g = GrafoManufatura(postos)
    g.adicionar_rota("SILO-100", "INSP-200", 0.8, "XV-101", "Alimentador Pneumático")
    g.adicionar_rota("INSP-200", "DIV-201",  1.2, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-201",  "CX-301",   0.5, "XV-201", "Pistão de Triagem 1")
    g.adicionar_rota("DIV-201",  "DIV-202",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-202",  "CX-302",   0.5, "XV-202", "Pistão de Triagem 2")
    g.adicionar_rota("DIV-202",  "DIV-203",  1.0, "M-101",  "Esteira Principal")
    g.adicionar_rota("DIV-203",  "CX-303",   0.5, "XV-203", "Pistão de Triagem 3")
    g.adicionar_rota("DIV-203",  "DESC-304", 0.8, "M-101",  "Calha de Descarte")
    return g

def criar_planta_fms_completa() -> GrafoManufatura:
    """Instancia a topologia expandida com rotas alternativas de bypass, contorno e recirculação."""
    g = criar_planta_fms_nominal()
    # Bypass expresso ligando o posto de inspeção diretamente ao desvio 3
    g.adicionar_rota("INSP-200", "DIV-203", 2.5, "XV-200B", "Bypass Rápido de Triagem")
    # Esteira auxiliar de contorno contornando o desvio 2 (DIV-201 -> DIV-203)
    g.adicionar_rota("DIV-201",  "DIV-203", 2.2, "M-101B",  "Esteira Auxiliar de Contorno")
    # Esteira de recirculação de refugos para reaproveitamento no silo
    g.adicionar_rota("DESC-304", "SILO-100", 3.5, "M-102",   "Esteira de Recirculação")
    return g

class RoteadorDijkstraFMS:
    """
    Motor de Roteamento Dinâmico em Tempo Real baseado no Algoritmo de Dijkstra com Min-Heap binário.
    """
    def __init__(self, grafo: GrafoManufatura):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str,
                               bloqueios: Optional[Set[str]] = None,
                               penalidades: Optional[Dict[Tuple[str, str], float]] = None) -> Tuple[float, List[str], List[str]]:
        """
        Calcula o caminho de menor custo métrico entre origem e destino via Dijkstra.
        Suporta bloqueio dinâmico de nós (bloqueios) e penalidades de congestionamento (penalidades).
        """
        if bloqueios is None:
            bloqueios = set()
        if penalidades is None:
            penalidades = {}
        if origem in bloqueios or destino in bloqueios:
            return float('inf'), [], []

        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]

        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]:
                continue
            if u == destino:
                break

            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso_base = self.g.adj_pesos[u_idx][v_idx]
                if peso_base < float('inf') and v not in bloqueios:
                    peso_efetivo = peso_base + penalidades.get((u, v), 0.0)
                    nova_d = d_u + peso_efetivo
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))

        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()

        if caminho and caminho[0] == origem:
            atuadores = [self.g.obter_atuador_aresta(caminho[i], caminho[i+1]) for i in range(len(caminho) - 1)]
            return dist[destino], caminho, atuadores
        return float('inf'), [], []

    def auditar_conectividade_scada(self, origem: str, destinos: List[str],
                                    bloqueios: Optional[Set[str]] = None,
                                    penalidades: Optional[Dict[Tuple[str, str], float]] = None) -> List[Dict[str, Any]]:
        """
        Auditoria periódica de roteamento em tempo de varredura para a IHM do SCADA.
        """
        if bloqueios is None:
            bloqueios = set()
        if penalidades is None:
            penalidades = {}

        relatorio = []
        for dst in destinos:
            custo, caminho, atuadores = self.calcular_menor_caminho(origem, dst, bloqueios, penalidades)
            if custo < float('inf'):
                status = "OPERACIONAL"
                if "XV-200B" in atuadores:
                    status = "CONTINGÊNCIA (Bypass)"
                elif "M-101B" in atuadores:
                    status = "CONTINGÊNCIA (Contorno)"

                relatorio.append({
                    "Destino": dst,
                    "Status SCADA": status,
                    "Distância (m)": f"{custo:.1f}",
                    "Saltos (Arestas)": len(caminho) - 1,
                    "Rota Ótima (Dijkstra)": " -> ".join(caminho),
                    "Atuadores Ativados": ", ".join(atuadores)
                })
            else:
                relatorio.append({
                    "Destino": dst,
                    "Status SCADA": "BLOQUEADO (Intertravamento)",
                    "Distância (m)": "∞",
                    "Saltos (Arestas)": "N/A",
                    "Rota Ótima (Dijkstra)": "Nenhum caminho viável",
                    "Atuadores Ativados": "INIBIÇÃO DE INFEED"
                })
        return relatorio

# --- INSTANCIAÇÃO DOS MODELOS ---
fms_nominal = criar_planta_fms_nominal()
fms_completa = criar_planta_fms_completa()

roteador_nominal = RoteadorDijkstraFMS(fms_nominal)
roteador_completo = RoteadorDijkstraFMS(fms_completa)
destinos_fms = ["CX-301", "CX-302", "CX-303", "DESC-304"]

# 1. Roteamento Ótimo Nominal via Dijkstra
print("=== 1. AUDITORIA DE ROTEAMENTO ÓTIMO NOMINAL VIA DIJKSTRA (FMS) ===")
relatorio_nom = roteador_nominal.auditar_conectividade_scada("SILO-100", destinos_fms)
print(formatar_tabela(relatorio_nom))

# 2. Reencaminhamento Dinâmico em Tempo Real sob Falha (DIV-202 Bloqueado)
print("\n=== 2. REENCAMINHAMENTO DINÂMICO SOB FALHA NO DESVIO 2 (DIV-202 BLOQUEADO) ===")
relatorio_falha = roteador_completo.auditar_conectividade_scada("SILO-100", destinos_fms, bloqueios={"DIV-202"})
print(formatar_tabela(relatorio_falha))

# 3. Roteamento sob Congestionamento / Penalidade Dinâmica de Peso
print("\n=== 3. ROTEAMENTO DINÂMICO SOB CONGESTIONAMENTO NO TRECHO PRINCIPAL (+1.5m em INSP->DIV1) ===")
penalidade_transito = {("INSP-200", "DIV-201"): 1.5}
relatorio_congest = roteador_completo.auditar_conectividade_scada("SILO-100", destinos_fms, penalidades=penalidade_transito)
print(formatar_tabela(relatorio_congest))

# 4. Comparativo Prático: BFS (Menos Atuadores) vs Dijkstra (Menor Distância Métrica)
print("\n=== 4. COMPARATIVO FORMAL DE ENGENHARIA: BFS vs DIJKSTRA (DIV-201 -> DIV-203) ===")
print("Análise do trecho DIV-201 até DIV-203:")
print("  Opção A (Esteira Principal): DIV-201 -> DIV-202 -> DIV-203 | Saltos = 2 | Comprimento = 1.0 + 1.0 = 2.0 m")
print("  Opção B (Esteira Contorno):  DIV-201 -> DIV-203             | Saltos = 1 | Comprimento = 2.2 m")
print("  -> Critério BFS (Aula 13): Escolhe Opção B (1 salto, menos atuadores mecânicos).")
print("  -> Critério Dijkstra (Aula 14): Escolhe Opção A (2.0 m < 2.2 m, menor distância física e tempo).")

# --- 5. ASSERÇÕES DETERMINÍSTICAS DE VALIDAÇÃO ---
# Validação das distâncias nominais
c_cx1, p_cx1, _ = roteador_nominal.calcular_menor_caminho("SILO-100", "CX-301")
assert c_cx1 == 2.5 and p_cx1 == ["SILO-100", "INSP-200", "DIV-201", "CX-301"]

c_cx2, p_cx2, _ = roteador_nominal.calcular_menor_caminho("SILO-100", "CX-302")
assert c_cx2 == 3.5

c_cx3_nom, p_cx3_nom, _ = roteador_nominal.calcular_menor_caminho("SILO-100", "CX-303")
assert c_cx3_nom == 4.5

# Validação do reencaminhamento via bypass na malha completa (3.8m < 4.5m)
c_cx3_comp, p_cx3_comp, at_cx3 = roteador_completo.calcular_menor_caminho("SILO-100", "CX-303")
assert c_cx3_comp == 3.8
assert "XV-200B" in at_cx3
assert p_cx3_comp == ["SILO-100", "INSP-200", "DIV-203", "CX-303"]

# Validação de bloqueio do desvio 2
c_falha_cx2, _, _ = roteador_completo.calcular_menor_caminho("SILO-100", "CX-302", bloqueios={"DIV-202"})
assert c_falha_cx2 == float('inf')

c_falha_cx3, p_falha_cx3, _ = roteador_completo.calcular_menor_caminho("SILO-100", "CX-303", bloqueios={"DIV-202"})
assert c_falha_cx3 == 3.8

print("\n[OK] Motor de Roteamento Dinâmico Dijkstra da FMS validado com sucesso!")

=== 1. AUDITORIA DE ROTEAMENTO ÓTIMO NOMINAL VIA DIJKSTRA (FMS) ===
Destino  | Status SCADA | Distância (m) | Saltos (Arestas) | Rota Ótima (Dijkstra)                                             | Atuadores Ativados                         
---------+--------------+---------------+------------------+-------------------------------------------------------------------+--------------------------------------------
CX-301   | OPERACIONAL  | 2.5           | 3                | SILO-100 -> INSP-200 -> DIV-201 -> CX-301                         | XV-101, M-101, XV-201                      
CX-302   | OPERACIONAL  | 3.5           | 4                | SILO-100 -> INSP-200 -> DIV-201 -> DIV-202 -> CX-302              | XV-101, M-101, M-101, XV-202               
CX-303   | OPERACIONAL  | 4.5           | 5                | SILO-100 -> INSP-200 -> DIV-201 -> DIV-202 -> DIV-203 -> CX-303   | XV-101, M-101, M-101, M-101, XV-203        
DESC-304 | OPERACIONAL  | 4.8           | 5                | SILO-1